# DataObject — Alias-Driven Structured Access to Sensor Data

The `DataObject` class bridges query metadata to time-series data with predictable, alias-based column names and built-in entity grouping.

**Before** (fragile positional indexing):
```python
df = query.latest_data(cast_value='float')
if df[0,1] > 75:  # what is column 1?
    ...
```

**After** (alias-driven):
```python
data = query.data(limit=1, order="desc", cast_value="float")
for basin_uri, group in data.by("basin"):
    cl = group["chlorine"]
    if cl["value"][0] > 75:
        ...
```

This notebook walks through the key features using the test CSV dataset.

## Setup

Connect to a running Acquirium server and load the test graph. Make sure containers are running (`make up` or `make testing-up`).

In [2]:
import time
from acquirium import Acquirium, DataObject
from acquirium.Client.query import Query
from acquirium.internals.internals_namespaces import ACQUIRIUM_NS

acq = Acquirium(server_url="localhost", server_port=8000, use_ssl=False)

# Load the test graph (CSV-backed sensors)
acq.insert_graph("../deployments/DPR/dpr-combined-model.ttl")

# Wait for ingestion to complete
time.sleep(1)
status = acq.client.ingest_status()
while status["done"] < status["total"] - status["error"]:
    time.sleep(2)
    status = acq.client.ingest_status()
print(f"Ingestion complete: {status}")

Ingestion complete: {'scheduled': 0, 'done': 10, 'error': 0, 'total': 10}


## 1. Basic Usage — `query.data()`

Call `.data()` on any query that has data nodes. This returns a `DataObject` instead of a raw DataFrame.

In [3]:
# Find all data points in the graph
query = acq.find_all_data()
query.metadata_head()

# Get a DataObject with all timeseries (first 10 rows per stream)
data = query.data(limit=10)
print(data)

                                              Metadata First 10 Rows                                               
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 0                                                    ┃ ext0                                                     ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ feed-pump-electric-current                           │ feed-pump-electric-current_pg_ref                        │
│ feed-pump-out-flow-rate                              │ feed-pump-out-flow-rate_pg_ref                           │
│ s2_valve-status                                      │ s2_valve-status_pg_ref                                   │
│ uv-led-status                                        │ uv-led-status_pg_ref                                     │
│ b1-valve-status                                      │ b1-valve-status_pg_ref                                   │
│ conn-b3w1_valve-to-uf_bypass_valve-ph                │ conn-b3w1_valve-to-uf_bypass_valve-ph_pg_ref             │
│ b3w1_valve-status                                    │ b3w1_valve-status_pg_ref                                 │
│ uf_backwash_pump-status                              │ uf_backwash_pump-status_pg_ref                           │
│ b3w2-valve-status                                    │ b3w2-valve-status_pg_ref                                 │
│ conn-chlorine-contactor-to-potable-effluent-valve-ph │ conn-chlorine-contactor-to-potable-effluent-valve-ph_pg… │
└──────────────────────────────────────────────────────┴──────────────────────────────────────────────────────────┘

DataObject(0 rows, aliases=[], entities=[])


## 2. Alias-Based Access — `data["alias"]`

Access time-series by the alias you gave the data node in your query. Returns a clean `[time, value]` DataFrame.

In [4]:
# See what aliases are available
print("Data aliases:", data.aliases)

# Access by alias name — returns [time, value]
first_alias = data.aliases[0]
df = data[first_alias]
print(f"\ndata['{first_alias}']:")
df

Data aliases: []


IndexError: list index out of range

## 3. Named Aliases with Entity + Data Queries

When you build a query with `find_entity` + `find_data`, you get meaningful alias names and entity context for grouping.

In [4]:
# Build a query: find entities of class B, then their related data
entity_query = (
    acq.find_entity(_class=ACQUIRIUM_NS.B, alias="equipment")
        .find_data(alias="sensor_data")
)
entity_query.show_query_graph()

# Fetch data with named aliases
data = entity_query.data(limit=5, order="desc", cast_value="float")
print(data)
print("\nData aliases:", data.aliases)
print("Entity aliases:", data.entity_aliases)

QUERY GRAPH

Nodes:
  0 [equipment]  class=urn:acquirium#B
  2 [sensor_data] [DATA]  class=*

Edges:
  equipment --(*, hops=1)--> sensor_data

Data nodes:
  2 [sensor_data]  filters={}

Current pointer: sensor_data

DataObject(20 rows, aliases=['sensor_data'], entities=['equipment'])

Data aliases: ['sensor_data']
Entity aliases: ['equipment']


In [5]:
# Access the named alias directly
data["sensor_data"]

time,value,ref_uri
"datetime[μs, UTC]",f64,str
2023-12-31 19:00:00 UTC,18.0,"""urn:ex/point_3_csv_ref"""
2023-12-31 19:00:00 UTC,44.0,"""urn:ex/point_2_csv_ref"""
2023-12-31 19:00:00 UTC,74.0,"""urn:ex/point_1_csv_ref"""
2023-12-31 19:00:00 UTC,50.429063,"""urn:ex/point_7_csv_ref"""
2023-12-31 20:00:00 UTC,84.0,"""urn:ex/point_3_csv_ref"""
…,…,…
2023-12-31 22:00:00 UTC,41.226018,"""urn:ex/point_7_csv_ref"""
2023-12-31 23:00:00 UTC,80.0,"""urn:ex/point_3_csv_ref"""
2023-12-31 23:00:00 UTC,85.0,"""urn:ex/point_2_csv_ref"""


## 4. Grouping by Entity — `data.by("alias")`

When your query includes entity nodes, you can group the data by entity. This is useful for "for each X, get its sensors" patterns.

In [16]:
# Group by the "equipment" entity
for entity_uri, group in data.by("equipment"):
    print(f"\nEquipment: {entity_uri}")
    print(f"  Aliases in group: {group.aliases}")
    sensor_df = group.dataframe(shape="wide")
    print(f"  Rows: {len(sensor_df)}")
    print(sensor_df)


Equipment: urn:ex/eq_2
  Aliases in group: ['sensor']
  Rows: 1
shape: (1, 4)
┌─────────────────────────┬──────────┬──────────┬──────────┐
│ time                    ┆ sensor_2 ┆ sensor_1 ┆ sensor_0 │
│ ---                     ┆ ---      ┆ ---      ┆ ---      │
│ datetime[μs, UTC]       ┆ f64      ┆ f64      ┆ f64      │
╞═════════════════════════╪══════════╪══════════╪══════════╡
│ 2023-12-31 23:00:00 UTC ┆ 80.0     ┆ 85.0     ┆ 50.0     │
└─────────────────────────┴──────────┴──────────┴──────────┘

Equipment: urn:ex/eq_7
  Aliases in group: ['sensor']
  Rows: 1
shape: (1, 2)
┌─────────────────────────┬──────────┐
│ time                    ┆ sensor   │
│ ---                     ┆ ---      │
│ datetime[μs, UTC]       ┆ f64      │
╞═════════════════════════╪══════════╡
│ 2023-12-31 23:00:00 UTC ┆ 42.33851 │
└─────────────────────────┴──────────┘


## 5. Flat DataFrames — `data.dataframe()`

Get a standard wide or narrow DataFrame when you need one for plotting or further analysis.

In [7]:
# Wide format: [time, alias_col_1, alias_col_2, ...]
wide_df = data.dataframe(shape="wide")
print("Wide DataFrame:")
wide_df

Wide DataFrame:


time,sensor_data_2,sensor_data_1,sensor_data_0,sensor_data_3
"datetime[μs, UTC]",f64,f64,f64,f64
2023-12-31 19:00:00 UTC,18.0,44.0,74.0,50.429063
2023-12-31 20:00:00 UTC,84.0,98.0,14.0,40.062884
2023-12-31 21:00:00 UTC,38.0,72.0,36.0,56.017775
2023-12-31 22:00:00 UTC,9.0,80.0,74.0,41.226018
2023-12-31 23:00:00 UTC,80.0,85.0,50.0,42.33851


In [8]:
# Narrow format: the full enriched tall frame
narrow_df = data.dataframe(shape="narrow")
print("Narrow DataFrame:")
narrow_df

Narrow DataFrame:


data_alias,point_uri,ref_uri,entity__equipment,time,value
str,str,str,str,"datetime[μs, UTC]",f64
"""sensor_data""","""urn:ex/point_3""","""urn:ex/point_3_csv_ref""","""urn:ex/eq_2""",2023-12-31 19:00:00 UTC,18.0
"""sensor_data""","""urn:ex/point_2""","""urn:ex/point_2_csv_ref""","""urn:ex/eq_2""",2023-12-31 19:00:00 UTC,44.0
"""sensor_data""","""urn:ex/point_1""","""urn:ex/point_1_csv_ref""","""urn:ex/eq_2""",2023-12-31 19:00:00 UTC,74.0
"""sensor_data""","""urn:ex/point_7""","""urn:ex/point_7_csv_ref""","""urn:ex/eq_7""",2023-12-31 19:00:00 UTC,50.429063
"""sensor_data""","""urn:ex/point_3""","""urn:ex/point_3_csv_ref""","""urn:ex/eq_2""",2023-12-31 20:00:00 UTC,84.0
…,…,…,…,…,…
"""sensor_data""","""urn:ex/point_7""","""urn:ex/point_7_csv_ref""","""urn:ex/eq_7""",2023-12-31 22:00:00 UTC,41.226018
"""sensor_data""","""urn:ex/point_3""","""urn:ex/point_3_csv_ref""","""urn:ex/eq_2""",2023-12-31 23:00:00 UTC,80.0
"""sensor_data""","""urn:ex/point_2""","""urn:ex/point_2_csv_ref""","""urn:ex/eq_2""",2023-12-31 23:00:00 UTC,85.0


## 6. Iterating Individual Series — `data.iter("alias")`

When you need to process each point's timeseries individually (e.g., per-sensor anomaly detection).

In [9]:
# Iterate over each individual point's timeseries
for point_uri, series_df in data.iter("sensor_data"):
    print(f"Point: {point_uri}  |  rows: {len(series_df)}  |  mean: {series_df['value'].mean():.2f}")

Point: urn:ex/point_1  |  rows: 5  |  mean: 49.60
Point: urn:ex/point_2  |  rows: 5  |  mean: 75.80
Point: urn:ex/point_3  |  rows: 5  |  mean: 45.80
Point: urn:ex/point_7  |  rows: 5  |  mean: 46.01


## 7. Metadata & Introspection

Inspect what's in the DataObject without looking at the raw timeseries.

In [10]:
# Metadata: unique combinations of alias, point_uri, ref_uri, and entity URIs
print("Metadata:")
data.metadata()

Metadata:


data_alias,point_uri,ref_uri,entity__equipment
str,str,str,str
"""sensor_data""","""urn:ex/point_2""","""urn:ex/point_2_csv_ref""","""urn:ex/eq_2"""
"""sensor_data""","""urn:ex/point_1""","""urn:ex/point_1_csv_ref""","""urn:ex/eq_2"""
"""sensor_data""","""urn:ex/point_3""","""urn:ex/point_3_csv_ref""","""urn:ex/eq_2"""
"""sensor_data""","""urn:ex/point_7""","""urn:ex/point_7_csv_ref""","""urn:ex/eq_7"""


In [11]:
# Reference info: which external refs back a given alias
print("Ref info for 'sensor_data':")
for idx, ref_uri in data.ref_info("sensor_data"):
    print(f"  [{idx}] {ref_uri}")

Ref info for 'sensor_data':
  [0] urn:ex/point_1_csv_ref
  [1] urn:ex/point_2_csv_ref
  [2] urn:ex/point_3_csv_ref
  [3] urn:ex/point_7_csv_ref


In [12]:
# Latest value for an alias
print("Latest value:")
data.latest("sensor_data")

Latest value:


time,value
"datetime[μs, UTC]",f64
2023-12-31 23:00:00 UTC,80.0


## 8. Multi-Level Query Example

A more complex query with multiple entity levels and data nodes, showing how grouping and alias access compose together.

In [13]:
# Multi-level: find B entities, their related E entities, and all their data
multi_query = (
    acq.find_entity(_class=ACQUIRIUM_NS.B, alias="parent")
       .find_related(_class=ACQUIRIUM_NS.E, alias="child")
       .find_all_data(alias="readings")
)
multi_query.show_query_graph()

multi_data = multi_query.data(limit=3, order="desc", cast_value="float")
print(multi_data)
print("\nEntity aliases:", multi_data.entity_aliases)

QUERY GRAPH

Nodes:
  0 [parent]  class=urn:acquirium#B
  2 [child]  class=urn:acquirium#E
  4 [readings] [DATA]  class=*
  6 [readings_1] [DATA]  class=*

Edges:
  parent --(*, hops=3)--> child
  parent --(*, hops=1)--> readings
  child --(*, hops=1)--> readings_1

Data nodes:
  4 [readings]  filters={}
  6 [readings_1]  filters={}

Current pointer: readings_1

DataObject(36 rows, aliases=['readings', 'readings_1'], entities=['child', 'parent'])

Entity aliases: ['child', 'parent']


In [14]:
# Group by child entity and summarize
if "child" in multi_data.entity_aliases:
    for child_uri, group in multi_data.by("child"):
        readings = group["readings"]
        if not readings.is_empty():
            print(f"Child: {child_uri}")
            print(f"  Latest value: {readings.sort('time', descending=True)['value'][0]}")
            print(f"  Mean: {readings['value'].mean():.2f}")
            print()

Child: urn:ex/eq_10
  Latest value: 80.0
  Mean: 55.30

Child: urn:ex/eq_5
  Latest value: 80.0
  Mean: 55.30



## 9. App Pattern — Threshold Alert (Before vs After)

The `DataObject` API makes app logic much more readable and robust.

In [15]:
# --- BEFORE: fragile positional indexing ---
# df = query.latest_data(cast_value='float')
# if df[0,1] > 75:   # What is column 1? What if column order changes?
#     print("Alert!")

# --- AFTER: alias-driven with DataObject ---
# Simulating the threshold app pattern with our test data
threshold_query = (
    acq.find_entity(_class=ACQUIRIUM_NS.B, alias="equipment")
       .find_data(alias="sensor")
)

data = threshold_query.data(limit=1, order="desc", cast_value="float")

for equip_uri, group in data.by("equipment"):
    sensor = group["sensor"]
    if sensor.is_empty():
        print(f"{equip_uri}: No data")
    else:
        val = sensor["value"][0]
        ts = sensor["time"][0]
        print(f"{equip_uri}: value={val:.2f} at {ts}")

urn:ex/eq_2: value=80.00 at 2023-12-31 23:00:00+00:00
urn:ex/eq_7: value=42.34 at 2023-12-31 23:00:00+00:00


## API Reference

| Method / Property | Returns | Description |
|---|---|---|
| `query.data(start, end, limit, order, cast_value)` | `DataObject` | Construct from a query |
| `data["alias"]` | `pl.DataFrame` | `[time, value]` for single-ref; `[time, value, ref_uri]` for multi-ref |
| `data.by("entity_alias")` | `Iterator[(str, DataObject)]` | Group by entity, yields `(uri, sub_DataObject)` |
| `data.dataframe(shape="wide")` | `pl.DataFrame` | Pivoted `[time, alias_1, alias_2, ...]` |
| `data.dataframe(shape="narrow")` | `pl.DataFrame` | Full enriched tall frame |
| `data.iter("alias")` | `Iterator[(str, pl.DataFrame)]` | Per-point `(point_uri, [time, value])` |
| `data.metadata()` | `pl.DataFrame` | Unique `(data_alias, point_uri, ref_uri, entity__*)` |
| `data.latest("alias")` | `pl.DataFrame` | Most recent `[time, value]` |
| `data.ref_info("alias")` | `list[(int, str)]` | Indexed ref URIs |
| `data.aliases` | `list[str]` | Available data aliases |
| `data.entity_aliases` | `list[str]` | Available entity aliases |
| `data.is_empty()` | `bool` | Whether any data exists |